In [1]:
from google.colab import drive
import pickle
import pandas as pd

# Mount Google Drive
drive.mount('/content/drive')

# TODO: replace with your actual path to the pkl file in Drive
pkl_path = '/content/drive/MyDrive/processed_eeg_dataset2.pkl'

with open(pkl_path, 'rb') as f:
    data = pickle.load(f)

epochs = data['epochs']
epoch_labels = data['epoch_labels']
epoch_trial_ids = data['epoch_trial_ids']

# Sanity checks
print(f"Number of epochs: {len(epochs)}")
print(f"Number of unique trials: {len(set(epoch_trial_ids))}")
print(f"Label distribution: {pd.Series(epoch_labels).value_counts()}")
print(f"Shape of one epoch: {epochs[0].shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Number of epochs: 5090
Number of unique trials: 1957
Label distribution: Left     2552
Right    2538
Name: count, dtype: int64
Shape of one epoch: (500, 5)


In [2]:
print(epochs[0].columns.tolist())

['Time', 'FZ', 'C3', 'CZ', 'C4']


In [3]:
from sklearn.model_selection import GroupKFold
import numpy as np

gkf = GroupKFold(n_splits=5)

epoch_labels_arr = np.array(epoch_labels)
epoch_trial_ids_arr = np.array(epoch_trial_ids)

folds = list(gkf.split(X=np.zeros(len(epoch_labels_arr)), y=epoch_labels_arr, groups=epoch_trial_ids_arr))

# Sanity check on fold 0
train_idx, test_idx = folds[0]
print(f"Fold 0 — train epochs: {len(train_idx)}, test epochs: {len(test_idx)}")

train_trials = set(epoch_trial_ids_arr[train_idx])
test_trials = set(epoch_trial_ids_arr[test_idx])
print(f"Overlap between train/test trials: {len(train_trials & test_trials)}")

Fold 0 — train epochs: 4072, test epochs: 1018
Overlap between train/test trials: 0


In [4]:
import pickle

with open("/content/drive/MyDrive/processed_eeg_dataset2.pkl", "rb") as f:
    data = pickle.load(f)

epoch_session_ids = data["epoch_session_ids"]
print("Loaded session ids:", len(epoch_session_ids))

Loaded session ids: 5090


In [5]:
from sklearn.preprocessing import StandardScaler
channel_cols = ['FZ', 'C3', 'CZ', 'C4']

def fit_transform_fold(train_idx, test_idx, epochs, channel_cols, epoch_session_ids):
    session_scalers = {}
    train_epochs_scaled = [None] * len(train_idx)
    test_epochs_scaled = [None] * len(test_idx)

    train_sessions = set(epoch_session_ids[i] for i in train_idx)
    for sess in train_sessions:
        sess_train_positions = [pos for pos, i in enumerate(train_idx) if epoch_session_ids[i] == sess]
        if len(sess_train_positions) == 0:
            continue
        sess_stack = np.vstack([epochs[train_idx[pos]][channel_cols].values for pos in sess_train_positions])
        scaler = StandardScaler()
        scaler.fit(sess_stack)
        session_scalers[sess] = scaler
        for pos in sess_train_positions:
            train_epochs_scaled[pos] = scaler.transform(epochs[train_idx[pos]][channel_cols].values)

    for pos, i in enumerate(test_idx):
        sess = epoch_session_ids[i]
        scaler = session_scalers.get(sess)
        if scaler is None:
            # test session wasn't seen in training; fall back to a scaler fit on all training data
            fallback = StandardScaler().fit(np.vstack([epochs[j][channel_cols].values for j in train_idx]))
            scaler = fallback
        test_epochs_scaled[pos] = scaler.transform(epochs[i][channel_cols].values)

    return train_epochs_scaled, test_epochs_scaled, session_scalers

# Test it on fold 0 only, to confirm it works before looping over all folds
train_epochs_scaled, test_epochs_scaled, scaler_fold0 = fit_transform_fold(
    train_idx, test_idx, epochs, channel_cols, epoch_session_ids
)

print(f"Train epochs scaled: {len(train_epochs_scaled)}, shape of one: {train_epochs_scaled[0].shape}")
print(f"Test epochs scaled: {len(test_epochs_scaled)}, shape of one: {test_epochs_scaled[0].shape}")

Train epochs scaled: 4072, shape of one: (500, 4)
Test epochs scaled: 1018, shape of one: (500, 4)


In [6]:
def reshape_for_eegnet(epoch_list):
    # each epoch: (500, 4) -> transpose to (4, 500) -> add dummy dim -> (1, 4, 500)
    reshaped = np.array([ep.T for ep in epoch_list])  # (n_epochs, 4, 500)
    reshaped = reshaped[:, np.newaxis, :, :]           # (n_epochs, 1, 4, 500)
    return reshaped

X_train_fold0 = reshape_for_eegnet(train_epochs_scaled)
X_test_fold0 = reshape_for_eegnet(test_epochs_scaled)

label_map = {'Left': 0, 'Right': 1}
y_train_fold0 = np.array([label_map[epoch_labels_arr[i]] for i in train_idx])
y_test_fold0 = np.array([label_map[epoch_labels_arr[i]] for i in test_idx])

print(f"X_train shape: {X_train_fold0.shape}")
print(f"X_test shape: {X_test_fold0.shape}")
print(f"y_train shape: {y_train_fold0.shape}, unique: {np.unique(y_train_fold0, return_counts=True)}")
print(f"y_test shape: {y_test_fold0.shape}, unique: {np.unique(y_test_fold0, return_counts=True)}")

X_train shape: (4072, 1, 4, 500)
X_test shape: (1018, 1, 4, 500)
y_train shape: (4072,), unique: (array([0, 1]), array([2023, 2049]))
y_test shape: (1018,), unique: (array([0, 1]), array([529, 489]))


In [7]:
import torch
import torch.nn as nn

class EEGNet(nn.Module):
    def __init__(self, n_channels=4, n_samples=500, n_classes=2, dropout=0.5):
        super(EEGNet, self).__init__()

        F1 = 8          # number of temporal filters
        D = 2           # depth multiplier for spatial filters
        F2 = F1 * D     # number of pointwise filters
        kernel_length = 64  # ~250ms at 250Hz, standard for mu/beta rhythms

        # Block 1: temporal conv -> depthwise spatial conv
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, kernel_size=(1, kernel_length), padding=(0, kernel_length // 2), bias=False),
            nn.BatchNorm2d(F1),
            nn.Conv2d(F1, F1 * D, kernel_size=(n_channels, 1), groups=F1, bias=False),  # depthwise
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout)
        )

        # Block 2: separable conv (depthwise + pointwise)
        self.block2 = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, kernel_size=(1, 16), padding=(0, 8), groups=F1 * D, bias=False),  # depthwise
            nn.Conv2d(F1 * D, F2, kernel_size=(1, 1), bias=False),  # pointwise
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(dropout)
        )

        # Figure out flattened size dynamically with a dummy forward pass
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            out = self.block2(self.block1(dummy))
            flat_size = out.view(1, -1).shape[1]

        self.classifier = nn.Linear(flat_size, n_classes)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

# Quick sanity check on fold 0 data
model = EEGNet(n_channels=4, n_samples=500, n_classes=2, dropout=0.5)
dummy_input = torch.tensor(X_train_fold0[:4], dtype=torch.float32)
output = model(dummy_input)
print(f"Output shape: {output.shape}")  # should be (4, 2)
print(model)

Output shape: torch.Size([4, 2])
EEGNet(
  (block1): Sequential(
    (0): Conv2d(1, 8, kernel_size=(1, 64), stride=(1, 1), padding=(0, 32), bias=False)
    (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): Conv2d(8, 16, kernel_size=(4, 1), stride=(1, 1), groups=8, bias=False)
    (3): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (4): ELU(alpha=1.0)
    (5): AvgPool2d(kernel_size=(1, 4), stride=(1, 4), padding=0)
    (6): Dropout(p=0.5, inplace=False)
  )
  (block2): Sequential(
    (0): Conv2d(16, 16, kernel_size=(1, 16), stride=(1, 1), padding=(0, 8), groups=16, bias=False)
    (1): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): ELU(alpha=1.0)
    (4): AvgPool2d(kernel_size=(1, 8), stride=(1, 8), padding=0)
    (5): Dropout(p=0.5, inplace=False)
  )
  (classifier): Linear(in_features=240, out_

In [8]:
from torch.utils.data import TensorDataset, DataLoader
import copy

def train_eegnet(X_train, y_train, X_test, y_test, n_channels=4, n_samples=500,
                  epochs=100, patience=50, batch_size=64, lr=5e-4):

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = EEGNet(n_channels=n_channels, n_samples=n_samples, dropout=0.25).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                              torch.tensor(y_train, dtype=torch.long))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test_t = torch.tensor(y_test, dtype=torch.long).to(device)

    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * X_batch.size(0)
            train_correct += (outputs.argmax(1) == y_batch).sum().item()
            train_total += X_batch.size(0)

        train_loss /= train_total
        train_acc = train_correct / train_total

        # Validation
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_test_t)
            val_loss = criterion(val_outputs, y_test_t).item()
            val_acc = (val_outputs.argmax(1) == y_test_t).float().mean().item()

        print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
              f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

        # Early stopping check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered at epoch {epoch+1}")
                break

    model.load_state_dict(best_model_state)
    return model, best_val_loss

# Run on fold 0 only first, to confirm training works before looping over all 5 folds
model_fold0, best_val_loss_fold0 = train_eegnet(X_train_fold0, y_train_fold0, X_test_fold0, y_test_fold0)

Epoch 1: train_loss=0.7134, train_acc=0.4993, val_loss=0.6984, val_acc=0.4902
Epoch 2: train_loss=0.7030, train_acc=0.5020, val_loss=0.6933, val_acc=0.5020
Epoch 3: train_loss=0.6951, train_acc=0.5091, val_loss=0.6928, val_acc=0.4853
Epoch 4: train_loss=0.6945, train_acc=0.5160, val_loss=0.6929, val_acc=0.5079
Epoch 5: train_loss=0.6946, train_acc=0.5098, val_loss=0.6936, val_acc=0.5167
Epoch 6: train_loss=0.6924, train_acc=0.5108, val_loss=0.6934, val_acc=0.5118
Epoch 7: train_loss=0.6932, train_acc=0.5145, val_loss=0.6927, val_acc=0.5059
Epoch 8: train_loss=0.6896, train_acc=0.5258, val_loss=0.6947, val_acc=0.4980
Epoch 9: train_loss=0.6887, train_acc=0.5214, val_loss=0.6933, val_acc=0.4882
Epoch 10: train_loss=0.6888, train_acc=0.5253, val_loss=0.6935, val_acc=0.5010
Epoch 11: train_loss=0.6874, train_acc=0.5359, val_loss=0.6949, val_acc=0.4951
Epoch 12: train_loss=0.6869, train_acc=0.5265, val_loss=0.6969, val_acc=0.4990
Epoch 13: train_loss=0.6868, train_acc=0.5233, val_loss=0.695

In [9]:
from sklearn.linear_model import LogisticRegression
from scipy.signal import welch

def band_power_features(epoch_scaled, fs=250, bands={'mu': (8,12), 'beta': (13,30)}):
    # epoch_scaled shape: (500, 4) -> channels FZ, C3, CZ, C4
    feats = []
    for ch in range(epoch_scaled.shape[1]):
        freqs, psd = welch(epoch_scaled[:, ch], fs=fs, nperseg=128)
        for band_name, (lo, hi) in bands.items():
            mask = (freqs >= lo) & (freqs <= hi)
            feats.append(psd[mask].mean())
    return feats

X_train_feats = np.array([band_power_features(ep) for ep in train_epochs_scaled])
X_test_feats = np.array([band_power_features(ep) for ep in test_epochs_scaled])

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_feats, y_train_fold0)
train_acc = clf.score(X_train_feats, y_train_fold0)
test_acc = clf.score(X_test_feats, y_test_fold0)

print(f"Baseline train_acc: {train_acc:.4f}, test_acc: {test_acc:.4f}")

Baseline train_acc: 0.5027, test_acc: 0.4794


In [11]:
# Overfit check: can EEGNet learn on a tiny, easy-to-memorize slice?
import numpy as np
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn

# Grab a small balanced subset
left_idx = [i for i, l in enumerate(epoch_labels) if l == 'Left'][:50]
right_idx = [i for i, l in enumerate(epoch_labels) if l == 'Right'][:50]
subset_idx = left_idx + right_idx

subset_epochs = [epochs[i][channel_cols].values for i in subset_idx]
subset_labels = np.array([0 if epoch_labels[i] == 'Left' else 1 for i in subset_idx])

# Scale (fit on this subset only, just for this diagnostic)
scaler = StandardScaler()
stacked = np.vstack(subset_epochs)
scaler.fit(stacked)
subset_scaled = [scaler.transform(e) for e in subset_epochs]

# Reshape for EEGNet: (n, 1, 4, 500)
X_subset = np.array([e.T for e in subset_scaled])[:, np.newaxis, :, :]
y_subset = subset_labels

print(f"X_subset shape: {X_subset.shape}, y_subset shape: {y_subset.shape}")
print(f"Class counts: {np.bincount(y_subset)}")

# Train to try to overfit
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = EEGNet(n_channels=4, n_samples=500, n_classes=2, dropout=0.0).to(device)  # dropout=0 to make overfitting easier
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

X_t = torch.tensor(X_subset, dtype=torch.float32).to(device)
y_t = torch.tensor(y_subset, dtype=torch.long).to(device)

for epoch in range(300):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_t)
    loss = criterion(outputs, y_t)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        acc = (outputs.argmax(1) == y_t).float().mean().item()
        print(f"Epoch {epoch+1}: loss={loss.item():.4f}, train_acc={acc:.4f}")

X_subset shape: (100, 1, 4, 500), y_subset shape: (100,)
Class counts: [50 50]
Epoch 20: loss=0.5420, train_acc=0.7300
Epoch 40: loss=0.4713, train_acc=0.7700
Epoch 60: loss=0.4141, train_acc=0.8300
Epoch 80: loss=0.3563, train_acc=0.8800
Epoch 100: loss=0.3038, train_acc=0.9100
Epoch 120: loss=0.2533, train_acc=0.9500
Epoch 140: loss=0.2086, train_acc=0.9500
Epoch 160: loss=0.1709, train_acc=0.9700
Epoch 180: loss=0.1390, train_acc=0.9900
Epoch 200: loss=0.1123, train_acc=1.0000
Epoch 220: loss=0.0900, train_acc=1.0000
Epoch 240: loss=0.0716, train_acc=1.0000
Epoch 260: loss=0.0576, train_acc=1.0000
Epoch 280: loss=0.0464, train_acc=1.0000
Epoch 300: loss=0.0377, train_acc=1.0000


In [12]:
# Global amplitude-based artifact filter (catches epochs a per-epoch relative check misses)
import numpy as np

channel_cols = ['FZ', 'C3', 'CZ', 'C4']

# Max absolute amplitude per epoch, across all channels
epoch_max_abs = np.array([epochs[i][channel_cols].abs().values.max() for i in range(len(epochs))])

# Global stats (NOT per-epoch, this is the key difference)
global_median = np.median(epoch_max_abs)
global_mad = np.median(np.abs(epoch_max_abs - global_median)) * 1.4826

threshold = global_median + 6 * global_mad

print(f"Global median max_abs: {global_median:.2f}")
print(f"Global MAD (scaled): {global_mad:.2f}")
print(f"Threshold: {threshold:.2f}")
print(f"Epochs above threshold: {(epoch_max_abs > threshold).sum()} / {len(epochs)} "
      f"({100 * (epoch_max_abs > threshold).mean():.1f}%)")

# Quick look at the distribution so you can sanity check the cutoff before committing
print(f"\nPercentiles of max_abs: 50th={np.percentile(epoch_max_abs, 50):.1f}, "
      f"90th={np.percentile(epoch_max_abs, 90):.1f}, "
      f"99th={np.percentile(epoch_max_abs, 99):.1f}, "
      f"max={epoch_max_abs.max():.1f}")

Global median max_abs: 43.20
Global MAD (scaled): 33.58
Threshold: 244.69
Epochs above threshold: 706 / 5090 (13.9%)

Percentiles of max_abs: 50th=43.2, 90th=491.1, 99th=5437.7, max=24790.1


In [13]:
# Apply the global amplitude filter, keeping epochs/labels/trial_ids in sync
keep_mask = epoch_max_abs <= threshold

epochs_clean = [epochs[i] for i in range(len(epochs)) if keep_mask[i]]
epoch_labels_clean = [epoch_labels[i] for i in range(len(epoch_labels)) if keep_mask[i]]
epoch_trial_ids_clean = [epoch_trial_ids[i] for i in range(len(epoch_trial_ids)) if keep_mask[i]]

print(f"Epochs before: {len(epochs)}")
print(f"Epochs after: {len(epochs_clean)}")

# Confirm class balance held up after filtering
import pandas as pd
print(pd.Series(epoch_labels_clean).value_counts())

# Overwrite the working variables so the rest of your pipeline (GroupKFold, scaling, reshape) uses the clean set
epochs = epochs_clean
epoch_labels = epoch_labels_clean
epoch_trial_ids = epoch_trial_ids_clean

Epochs before: 5090
Epochs after: 4384
Left     2203
Right    2181
Name: count, dtype: int64


In [14]:
from collections import defaultdict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

# 1. Reconstruct each epoch's position within its trial (0 = first 2s window, 1 = next, etc)
position_counter = defaultdict(int)
epoch_positions = []
for tid in epoch_trial_ids:
    epoch_positions.append(position_counter[tid])
    position_counter[tid] += 1
epoch_positions = np.array(epoch_positions)

print("Epochs per position:")
print(pd.Series(epoch_positions).value_counts().sort_index())

# 2. For each position, extract band-power features and test with cross-validation
label_map = {'Left': 0, 'Right': 1}
y_all = np.array([label_map[l] for l in epoch_labels])

for pos in sorted(set(epoch_positions)):
    idx = np.where(epoch_positions == pos)[0]
    if len(idx) < 20:
        print(f"Position {pos}: only {len(idx)} epochs, skipping")
        continue

    X_pos = np.array([band_power_features(epochs[i][channel_cols].values) for i in idx])
    y_pos = y_all[idx]

    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    scores = cross_val_score(clf, X_pos, y_pos, cv=StratifiedKFold(5, shuffle=True, random_state=42))

    print(f"Position {pos}: n={len(idx)}, mean CV accuracy={scores.mean():.3f} (+/- {scores.std():.3f})")

Epochs per position:
0    1782
1    1490
2    1069
3      39
4       4
Name: count, dtype: int64
Position 0: n=1782, mean CV accuracy=0.479 (+/- 0.013)
Position 1: n=1490, mean CV accuracy=0.496 (+/- 0.014)
Position 2: n=1069, mean CV accuracy=0.514 (+/- 0.037)
Position 3: n=39, mean CV accuracy=0.461 (+/- 0.051)
Position 4: only 4 epochs, skipping


In [15]:
from scipy import stats

bands = {'mu': (8, 12), 'beta': (13, 30)}
channel_cols = ['FZ', 'C3', 'CZ', 'C4']

left_mask = np.array(epoch_labels) == 'Left'
right_mask = np.array(epoch_labels) == 'Right'

print("=== Per-channel, per-band univariate test (Left vs Right) ===")
band_power_cache = {}
for ch in channel_cols:
    for band_name, (lo, hi) in bands.items():
        powers = []
        for e in all_epochs if 'all_epochs' in dir() else epochs:
            f, pxx = welch(e[ch].values, fs=250, nperseg=128)
            mask = (f >= lo) & (f <= hi)
            powers.append(pxx[mask].mean())
        powers = np.array(powers)
        band_power_cache[(ch, band_name)] = powers

        t_stat, p_val = stats.ttest_ind(powers[left_mask], powers[right_mask])
        mean_diff = powers[left_mask].mean() - powers[right_mask].mean()
        print(f"{ch:3s} {band_name:4s}: mean_diff={mean_diff:+.3f}, t={t_stat:+.2f}, p={p_val:.4f}")

print("\n=== Lateralization index: C3 - C4 mu power ===")
lat_index = band_power_cache[('C3', 'mu')] - band_power_cache[('C4', 'mu')]
t_stat, p_val = stats.ttest_ind(lat_index[left_mask], lat_index[right_mask])
print(f"Left mean: {lat_index[left_mask].mean():.3f}, Right mean: {lat_index[right_mask].mean():.3f}")
print(f"t={t_stat:+.2f}, p={p_val:.4f}")

=== Per-channel, per-band univariate test (Left vs Right) ===
FZ  mu  : mean_diff=-0.017, t=-0.17, p=0.8653
FZ  beta: mean_diff=-0.000, t=-0.09, p=0.9244
C3  mu  : mean_diff=+0.026, t=+0.29, p=0.7687
C3  beta: mean_diff=+0.005, t=+0.46, p=0.6488
CZ  mu  : mean_diff=-0.019, t=-0.67, p=0.5024
CZ  beta: mean_diff=-0.007, t=-1.54, p=0.1245
C4  mu  : mean_diff=+0.010, t=+0.11, p=0.9152
C4  beta: mean_diff=+0.007, t=+0.38, p=0.7029

=== Lateralization index: C3 - C4 mu power ===
Left mean: 0.023, Right mean: 0.007
t=+0.20, p=0.8418


In [16]:
import glob
import os
import re

# Rebuild the sorted file list directly (adjust EEG_FOLDER path if needed)
EEG_FOLDER = "MI CSV"  # change this to match your actual folder path

eeg_files = glob.glob(os.path.join(EEG_FOLDER, "*.csv"))

def extract_number(path):
    match = re.search(r"(\d+)\.csv$", os.path.basename(path))
    return int(match.group(1)) if match else None

eeg_files_sorted = sorted(eeg_files, key=extract_number)

print("Files found:", len(eeg_files_sorted))
print("First few:", [os.path.basename(f) for f in eeg_files_sorted[:3]])
print("Last few:", [os.path.basename(f) for f in eeg_files_sorted[-3:]])

Files found: 0
First few: []
Last few: []


In [17]:
import pickle
import numpy as np
from sklearn.model_selection import GroupKFold

with open("/content/drive/MyDrive/processed_eeg_dataset2.pkl", "rb") as f:
    data = pickle.load(f)

epochs = data["epochs"]
epoch_labels = data["epoch_labels"]
epoch_trial_ids = data["epoch_trial_ids"]
epoch_session_ids = data["epoch_session_ids"]

print("epochs:", len(epochs))
print("epoch_labels:", len(epoch_labels))
print("epoch_trial_ids:", len(epoch_trial_ids))
print("epoch_session_ids:", len(epoch_session_ids))
# all four of these should print the same number

label_map = {'Left': 0, 'Right': 1}
y_all = np.array([label_map[l] for l in epoch_labels])
session_groups = np.array(epoch_session_ids)

print("y_all:", len(y_all))
print("session_groups:", len(session_groups))

gkf_session = GroupKFold(n_splits=5)
fold_indices = list(gkf_session.split(np.zeros(len(y_all)), y_all, groups=session_groups))
train_idx, test_idx = fold_indices[0]

print("\nTrain epochs:", len(train_idx))
print("Test epochs:", len(test_idx))
print("Train sessions:", len(set(session_groups[train_idx])))
print("Test sessions:", len(set(session_groups[test_idx])))
print("Overlap (should be 0):", len(set(session_groups[train_idx]) & set(session_groups[test_idx])))

epochs: 5090
epoch_labels: 5090
epoch_trial_ids: 5090
epoch_session_ids: 5090
y_all: 5090
session_groups: 5090

Train epochs: 4065
Test epochs: 1025
Train sessions: 32
Test sessions: 8
Overlap (should be 0): 0


In [18]:
from scipy.signal import welch
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def band_power_features(epoch_scaled, fs=250, bands={'mu': (8,12), 'beta': (13,30)}):
    feats = []
    for ch in range(epoch_scaled.shape[1]):
        freqs, psd = welch(epoch_scaled[:, ch], fs=fs, nperseg=128)
        for band_name, (lo, hi) in bands.items():
            mask = (freqs >= lo) & (freqs <= hi)
            feats.append(psd[mask].mean())
    return feats

channel_cols = ['FZ', 'C3', 'CZ', 'C4']

# Raw band power features for everyone
X_raw = np.array([band_power_features(epochs[i][channel_cols].values) for i in range(len(epochs))])

# Per-session normalization, fit only on stats from that session's TRAIN portion
X_normalized = np.zeros_like(X_raw)
train_sessions = set(session_groups[train_idx])

for sess in set(session_groups):
    sess_all_idx = np.where(session_groups == sess)[0]
    sess_train_idx = np.intersect1d(sess_all_idx, train_idx)

    if len(sess_train_idx) >= 2:
        mean = X_raw[sess_train_idx].mean(axis=0)
        std = X_raw[sess_train_idx].std(axis=0)
    else:
        # session has no training data (fully held out) - fall back to global train stats
        mean = X_raw[train_idx].mean(axis=0)
        std = X_raw[train_idx].std(axis=0)

    std[std == 0] = 1
    X_normalized[sess_all_idx] = (X_raw[sess_all_idx] - mean) / std

clf = LogisticRegression(max_iter=1000)
clf.fit(X_normalized[train_idx], y_all[train_idx])
test_acc = clf.score(X_normalized[test_idx], y_all[test_idx])
print(f"Held-out-session test accuracy: {test_acc:.3f}")

Held-out-session test accuracy: 0.488


In [19]:
from sklearn.model_selection import GroupKFold

# Back to trial-based grouping (sessions can appear in both train and test)
gkf_trial = GroupKFold(n_splits=5)
trial_groups = np.array(epoch_trial_ids)

fold_indices = list(gkf_trial.split(np.zeros(len(y_all)), y_all, groups=trial_groups))
train_idx, test_idx = fold_indices[0]

print("Train epochs:", len(train_idx), " Test epochs:", len(test_idx))

# Per-session normalization, fit only on each session's TRAIN portion (same logic as before)
X_normalized2 = np.zeros_like(X_raw)

for sess in set(session_groups):
    sess_all_idx = np.where(session_groups == sess)[0]
    sess_train_idx = np.intersect1d(sess_all_idx, train_idx)

    if len(sess_train_idx) >= 2:
        mean = X_raw[sess_train_idx].mean(axis=0)
        std = X_raw[sess_train_idx].std(axis=0)
    else:
        mean = X_raw[train_idx].mean(axis=0)
        std = X_raw[train_idx].std(axis=0)

    std[std == 0] = 1
    X_normalized2[sess_all_idx] = (X_raw[sess_all_idx] - mean) / std

clf = LogisticRegression(max_iter=1000)
clf.fit(X_normalized2[train_idx], y_all[train_idx])
test_acc = clf.score(X_normalized2[test_idx], y_all[test_idx])
print(f"Trial-split test accuracy (per-session normalized): {test_acc:.3f}")

Train epochs: 4072  Test epochs: 1018
Trial-split test accuracy (per-session normalized): 0.483


In [20]:
import numpy as np
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

np.random.seed(0)
real_scores = []
shuffled_scores = []

for sess in sorted(set(session_groups)):
    idx = np.where(session_groups == sess)[0]
    y_sess = y_all[idx]
    if len(idx) < 30 or len(set(y_sess)) < 2:
        continue

    X_sess = X_raw[idx]

    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

    # Real labels
    real = cross_val_score(clf, X_sess, y_sess, cv=StratifiedKFold(3, shuffle=True, random_state=42))
    real_scores.append(real.mean())

    # Shuffled labels (destroys any real relationship, keeps class balance)
    y_shuffled = np.random.permutation(y_sess)
    shuffled = cross_val_score(clf, X_sess, y_shuffled, cv=StratifiedKFold(3, shuffle=True, random_state=42))
    shuffled_scores.append(shuffled.mean())

real_scores = np.array(real_scores)
shuffled_scores = np.array(shuffled_scores)

print(f"Real labels    - mean: {real_scores.mean():.3f}, std: {real_scores.std():.3f}, max: {real_scores.max():.3f}")
print(f"Shuffled labels - mean: {shuffled_scores.mean():.3f}, std: {shuffled_scores.std():.3f}, max: {shuffled_scores.max():.3f}")

from scipy import stats
t_stat, p_val = stats.ttest_rel(real_scores, shuffled_scores)
print(f"\nPaired t-test (real vs shuffled per session): t={t_stat:.2f}, p={p_val:.4f}")

Real labels    - mean: 0.547, std: 0.063, max: 0.695
Shuffled labels - mean: 0.548, std: 0.057, max: 0.683

Paired t-test (real vs shuffled per session): t=-0.17, p=0.8665


-------------------------

In [21]:
import pickle
import numpy as np

with open("/content/drive/MyDrive/processed_eeg_dataset2.pkl", "rb") as f:
    data = pickle.load(f)

epochs_full = data["epochs"]
epoch_labels_full = data["epoch_labels"]
epoch_trial_ids_full = data["epoch_trial_ids"]
epoch_session_ids_full = data["epoch_session_ids"]

channel_cols = ['FZ', 'C3', 'CZ', 'C4']

# same global amplitude filter as before, applied to the full aligned set
epoch_max_abs = np.array([epochs_full[i][channel_cols].abs().values.max() for i in range(len(epochs_full))])
global_median = np.median(epoch_max_abs)
global_mad = np.median(np.abs(epoch_max_abs - global_median)) * 1.4826
threshold = global_median + 6 * global_mad
keep_mask = epoch_max_abs <= threshold

epochs_dl = [epochs_full[i] for i in range(len(epochs_full)) if keep_mask[i]]
labels_dl = [epoch_labels_full[i] for i in range(len(epoch_labels_full)) if keep_mask[i]]
trial_ids_dl = np.array([epoch_trial_ids_full[i] for i in range(len(epoch_trial_ids_full)) if keep_mask[i]])
session_ids_dl = np.array([epoch_session_ids_full[i] for i in range(len(epoch_session_ids_full)) if keep_mask[i]])

print(f"Epochs after filter: {len(epochs_dl)}")
print(f"Sessions: {len(set(session_ids_dl))}")

Epochs after filter: 4384
Sessions: 40


In [22]:
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
import numpy as np

# fixed version: fallback scaler computed once, not per test epoch
def fit_transform_fold(train_idx, test_idx, epochs, channel_cols, epoch_session_ids):
    session_scalers = {}
    train_epochs_scaled = [None] * len(train_idx)
    test_epochs_scaled = [None] * len(test_idx)

    train_sessions = set(epoch_session_ids[i] for i in train_idx)
    for sess in train_sessions:
        sess_train_positions = [pos for pos, i in enumerate(train_idx) if epoch_session_ids[i] == sess]
        if len(sess_train_positions) == 0:
            continue
        sess_stack = np.vstack([epochs[train_idx[pos]][channel_cols].values for pos in sess_train_positions])
        scaler = StandardScaler()
        scaler.fit(sess_stack)
        session_scalers[sess] = scaler
        for pos in sess_train_positions:
            train_epochs_scaled[pos] = scaler.transform(epochs[train_idx[pos]][channel_cols].values)

    # compute the fallback scaler ONCE, not per test epoch
    fallback_scaler = StandardScaler().fit(np.vstack([epochs[j][channel_cols].values for j in train_idx]))

    for pos, i in enumerate(test_idx):
        sess = epoch_session_ids[i]
        scaler = session_scalers.get(sess, fallback_scaler)
        test_epochs_scaled[pos] = scaler.transform(epochs[i][channel_cols].values)

    return train_epochs_scaled, test_epochs_scaled, session_scalers


label_map = {'Left': 0, 'Right': 1}
y_dl = np.array([label_map[l] for l in labels_dl])

gkf = GroupKFold(n_splits=5)
fold_indices = list(gkf.split(np.zeros(len(y_dl)), y_dl, groups=session_ids_dl))
train_idx, test_idx = fold_indices[0]

print(f"Train epochs: {len(train_idx)}, Test epochs: {len(test_idx)}")
print(f"Train sessions: {len(set(session_ids_dl[train_idx]))}, Test sessions: {len(set(session_ids_dl[test_idx]))}")
print(f"Session overlap (should be 0): {len(set(session_ids_dl[train_idx]) & set(session_ids_dl[test_idx]))}")

train_scaled, test_scaled, _ = fit_transform_fold(train_idx, test_idx, epochs_dl, channel_cols, session_ids_dl)

X_train_dl = reshape_for_eegnet(train_scaled)
X_test_dl = reshape_for_eegnet(test_scaled)
y_train_dl = y_dl[train_idx]
y_test_dl = y_dl[test_idx]

print(f"X_train: {X_train_dl.shape}, X_test: {X_test_dl.shape}")

Train epochs: 3505, Test epochs: 879
Train sessions: 32, Test sessions: 8
Session overlap (should be 0): 0
X_train: (3505, 1, 4, 500), X_test: (879, 1, 4, 500)


In [23]:
import torch
import torch.nn as nn

def quick_train_eegnet(X_train, y_train, X_test, y_test, n_epochs=40, seed=0):
    torch.manual_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = EEGNet(n_channels=4, n_samples=X_train.shape[-1], n_classes=2, dropout=0.5).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.long).to(device)
    X_te = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_te = torch.tensor(y_test, dtype=torch.long).to(device)

    for epoch in range(n_epochs):
        model.train()
        optimizer.zero_grad()
        out = model(X_tr)
        loss = criterion(out, y_tr)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        train_acc = (model(X_tr).argmax(1) == y_tr).float().mean().item()
        test_acc = (model(X_te).argmax(1) == y_te).float().mean().item()

    return train_acc, test_acc

In [24]:
import numpy as np

# real labels
real_train_acc, real_test_acc = quick_train_eegnet(X_train_dl, y_train_dl, X_test_dl, y_test_dl)
print(f"Real labels    - train_acc: {real_train_acc:.3f}, test_acc: {real_test_acc:.3f}")

# shuffled labels: permute globally, keeps class balance, destroys any real relationship
np.random.seed(0)
y_train_shuffled = np.random.permutation(y_train_dl)
y_test_shuffled = np.random.permutation(y_test_dl)

shuf_train_acc, shuf_test_acc = quick_train_eegnet(X_train_dl, y_train_shuffled, X_test_dl, y_test_shuffled)
print(f"Shuffled labels - train_acc: {shuf_train_acc:.3f}, test_acc: {shuf_test_acc:.3f}")

Real labels    - train_acc: 0.565, test_acc: 0.527
Shuffled labels - train_acc: 0.546, test_acc: 0.457


In [ ]:
import numpy as np

n_folds_to_test = 3   # how many of the 5 GroupKFold folds to check
n_seeds = 5           # repeats per fold, to average out random-init noise

real_scores = []
shuffled_scores = []

for fold_i in range(n_folds_to_test):
    train_idx, test_idx = fold_indices[fold_i]

    train_scaled, test_scaled, _ = fit_transform_fold(train_idx, test_idx, epochs_dl, channel_cols, session_ids_dl)
    X_train_fold = reshape_for_eegnet(train_scaled)
    X_test_fold = reshape_for_eegnet(test_scaled)
    y_train_fold = y_dl[train_idx]
    y_test_fold = y_dl[test_idx]

    for seed in range(n_seeds):
        _, real_test_acc = quick_train_eegnet(X_train_fold, y_train_fold, X_test_fold, y_test_fold, seed=seed)
        real_scores.append(real_test_acc)

        rng = np.random.RandomState(seed)
        y_train_shuf = rng.permutation(y_train_fold)
        y_test_shuf = rng.permutation(y_test_fold)
        _, shuf_test_acc = quick_train_eegnet(X_train_fold, y_train_shuf, X_test_fold, y_test_shuf, seed=seed)
        shuffled_scores.append(shuf_test_acc)

    print(f"Fold {fold_i} done")

real_scores = np.array(real_scores)
shuffled_scores = np.array(shuffled_scores)

print(f"\nReal      - mean: {real_scores.mean():.3f}, std: {real_scores.std():.3f}")
print(f"Shuffled  - mean: {shuffled_scores.mean():.3f}, std: {shuffled_scores.std():.3f}")

from scipy import stats
t_stat, p_val = stats.ttest_ind(real_scores, shuffled_scores)
print(f"\nt={t_stat:.2f}, p={p_val:.4f}")

In [10]:
import torch
import json

# Save model weights
torch.save(model.state_dict(), "/content/eegnet_model.pth")

# Save architecture config so you can rebuild the exact same model shape later
model_config = {
    "n_channels": 4,
    "n_samples": 500,
    "n_classes": 2,
    "dropout": 0.5,
    "channel_order": ["FZ", "C3", "CZ", "C4"],
    "label_map": {"Left": 0, "Right": 1}
}

with open("/content/eegnet_config.json", "w") as f:
    json.dump(model_config, f, indent=2)

print("Saved model weights to eegnet_model.pth")
print("Saved config to eegnet_config.json")

Saved model weights to eegnet_model.pth
Saved config to eegnet_config.json
